In [ ]:
import sys

# Self-discover the deployed libs path from this notebook's own location.
# When deployed by the bundle, this notebook lives at <bundle-root>/files/setup/,
# so the libs folder is a sibling: <bundle-root>/files/libs/. The widget
# remains so an explicit value can still override the default.
try:
    _nb_path = (
        dbutils.notebook.entry_point.getDbutils()
            .notebook().getContext().notebookPath().get()
    )
    if "/files/" in _nb_path:
        _default_lib = _nb_path.split("/files/")[0] + "/files/libs"
        if not _default_lib.startswith("/Workspace"):
            _default_lib = "/Workspace" + _default_lib
    else:
        _default_lib = ""
except Exception:
    _default_lib = ""

dbutils.widgets.text("shared_lib_path", _default_lib)
sys.path.insert(0, dbutils.widgets.get("shared_lib_path"))

dbutils.widgets.text("catalog", "vinoworld")
CATALOG = dbutils.widgets.get("catalog")

In [ ]:
# ---------------------------------------------------------------------------
# seed_volumes: copies data files from the reference catalog into a newly
# provisioned target catalog so the pipeline can run without manual uploads.
#
# Skipped automatically when the target IS the reference catalog (prod).
# Idempotent: files already present in the target are never overwritten.
# Archive subdirectories are not copied — only active data files.
# ---------------------------------------------------------------------------

SOURCE_CATALOG = "vinoworld"   # always seed from the reference/prod catalog

VOLUME_NAMES = ["arancione", "celeste", "verde", "productdata", "masterdata"]

if CATALOG == SOURCE_CATALOG:
    print(f"Target catalog is '{CATALOG}' (production). No seeding needed — exiting cleanly.")
    dbutils.notebook.exit("skipped: target is production catalog")

In [ ]:
print(f"Seeding volumes: '{SOURCE_CATALOG}' → '{CATALOG}'\n")

copied_total  = 0
skipped_total = 0
errors        = []

try:
    for vol in VOLUME_NAMES:
        source_base = f"/Volumes/{SOURCE_CATALOG}/datafiles/{vol}/"
        target_base = f"/Volumes/{CATALOG}/datafiles/{vol}/"

        try:
            source_files = dbutils.fs.ls(source_base)
        except Exception:
            print(f"  [{vol}] Source volume not found or empty — skipping")
            continue

        # Build set of filenames already present in the target (idempotency)
        try:
            existing = {f.name for f in dbutils.fs.ls(target_base)}
        except Exception:
            existing = set()

        copied  = 0
        skipped = 0

        for f in source_files:
            if f.path.endswith("/"):   # skip subdirectories (e.g. archive/)
                continue
            if f.name in existing:
                skipped += 1
            else:
                try:
                    dbutils.fs.cp(f.path, target_base + f.name)
                    copied += 1
                except Exception as e:
                    errors.append(f"{vol}/{f.name}: {e}")

        print(f"  [{vol}] {copied} file(s) copied, {skipped} already present")
        copied_total  += copied
        skipped_total += skipped

except dbutils.NotebookExit:
    raise

except Exception as e:
    raise

if errors:
    raise RuntimeError(
        f"{len(errors)} file(s) failed to copy:\n" + "\n".join(errors)
    )

print(f"\nDone. {copied_total} file(s) copied, {skipped_total} already present.")